# 02b · Test B: one random 80:20 borehole split (comparison only)

The same setup as the friend's Phnom Penh notebook: a random 20 % of the development boreholes is the test set; the other 80 % are used for training, with Optuna tuning on 5 random borehole groups inside them. One split, fixed seed, the same for every target and model.

Test boreholes usually have a training borehole nearby, so this test is **easier** than Test A. Use it only to compare with the friend's R²; model choice and the thesis numbers come from Test A (notebook 03).

**Step 1, pilot:** `PILOT = True` runs the first target with XGB. **Step 2:** `PILOT = False` runs everything; it saves after every run and continues after a stop.

**Run time (estimate):** about a quarter of Test A, under 1 hour for all 6 targets.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])
import joblib
from tqdm import tqdm

PILOT = True           # True: first target, XGB only.  False: everything.
RUN_TARGETS = None     # e.g. ["SPT_N"]

runs = H.run_list(S, test="random")
if PILOT:
    runs = [r for r in runs if r["model"] == "XGB"][:1]
elif RUN_TARGETS:
    runs = [r for r in runs if r["target"] in RUN_TARGETS]
PRIV = H.out_dir(S, "02b_random_split", private=True)
(PRIV / "checkpoints").mkdir(exist_ok=True)
dev = {t: pd.read_csv(H.processed_dir(S) / f"samples_{t}.csv").query("set == 'development'") for t in {r["target"] for r in runs}}
test_codes, groups = H.random_split(S)
print(f"Random split: {len(test_codes)} test boreholes, {len(groups)} training boreholes (all targets together).")

## Run (or continue)

In [ ]:
for r in tqdm(runs, desc="runs", mininterval=10):
    ck = PRIV / "checkpoints" / f"{r['tag']}.joblib"
    if ck.exists():
        continue
    res = H.run_random(r, dev[r["target"]], S)
    res.pop("model")
    joblib.dump(res, ck, compress=3)
    sc = H.scores(res["predictions"].y, res["predictions"].y_pred)
    tqdm.write(f"{r['tag']:<30} RMSE {sc['rmse']:.2f}  R2 {sc['r2']:.3f}  ({res['info']['seconds'] / 60:.1f} min)")
print(f"Done ({(time.time() - T0) / 3600:.2f} h in this session).")

## Collect the results (after the full run)

In [ ]:
if not PILOT:
    preds, infos = [], []
    for f in sorted((PRIV / "checkpoints").glob("*.joblib")):
        res = joblib.load(f)
        preds.append(res["predictions"]); infos.append(res["info"])
    preds = pd.concat(preds, ignore_index=True)
    preds.to_csv(PRIV / "random_predictions.csv", index=False)
    pd.DataFrame(infos).drop(columns=["params"]).to_csv(H.out_dir(S, "02b_random_split") / "run_info.csv", index=False)
    print(f"{len(preds)} test predictions saved. Next: notebook 03.")